# 02 — Entraînement GPU (`w2v-bert-2.0` 16L) + Export ONNX INT8 + 1ère Soumission Zindi

**Réglages Kaggle requis :**
- **Accelerator** : `GPU T4 x2` ou `GPU P100`
- **Internet** : `ON`
- **Données CSV** : si `Train_reliable.csv`, `Train_religious.csv`, `Test.csv`, `SampleSubmission.csv` ne sont pas dans le bucket GCS, ajoute-les via **`+ Add Input → Upload`** (10 Mo au total).

**Ce que fait ce notebook de bout en bout :**
1. Télécharge toutes les archives audio `reliable` (~106 h) + `religious` (~40 h) + `test` (~22 h) depuis `https://storage.googleapis.com/fongbe/`.
2. Construit le split miroir (`train_clean` = 138 h, `val_mirror` = 2 396 général + 1 194 livre `JDG`).
3. Entraîne `facebook/w2v-bert-2.0` **16 couches** (MIT, SpecAugment sans pitch-shift).
4. Exporte le meilleur checkpoint en **ONNX INT8** + **encodeur isolé**.
5. Vérifie la conformité officielle avec `measure_frugality.py` (1 CPU, offline) et calcule **`TSER_mirror`** sur les 3 590 phrases.
6. Génère **`submission_w2vbert_16L_clean.csv`** strictement depuis le modèle **ONNX INT8** passé par `syllabify()`.

In [ ]:
# 0. Paramètres principaux (modifiables ici)
from pathlib import Path

NUM_LAYERS = 16            # 16L validé frugal (415 MB, RTF 0.58, RAM 1.47 GB). Mettre 24 pour Teacher complet.
MAX_STEPS = 3000           # ~4.7h sur Kaggle T4 (eff_batch=32 -> 1.5 epoch sur 138 h -> ~13.85% TSER_mirror)
EVAL_STEPS = 500           # Évaluation mirror pendant l'entraînement
EVAL_LIMIT = 600           # Sous-ensemble stratifié pendant le training (éval complète 3 590 utts à la fin)
BATCH_SIZE = 4             # Garder 4 sur T4 16 Go
GRAD_ACCUM = 8             # Effective batch size = 32
LEARNING_RATE = 5e-5
MAX_SECONDS = 16.0
SPEC_AUGMENT = True        # SpecAugment sans pitch-shift (préserve les tons !)
VAL_BOOK = "JDG"           # Livre biblique tenu de côté (miroir exact 67% gen / 33% rel)

# Archives à télécharger depuis https://storage.googleapis.com/fongbe/ (0 = TOUTES)
N_RELIABLE_ARCHIVES = 0    # 0 = toutes les archives reliable (~106 h)
N_RELIGIOUS_ARCHIVES = 0   # 0 = toutes les archives religious (~40 h)
DOWNLOAD_TEST = True       # Télécharger audio_test_*.tar pour produire submission.csv

# Reprise éventuelle d'une session précédente (ex: "/kaggle/input/mon-ckpt/latest")
RESUME_FROM = None

# Inférence finale sur Test.csv (ONNX INT8 bit-exact avec measure_frugality.py)
RUN_TEST_INFERENCE = True
INFER_WORKERS = 4          # 4 workers mono-thread en parallèle sur Kaggle CPU (~3h au lieu de 12h, bit-exact !)

BUCKET_URL = "https://storage.googleapis.com/fongbe"
if Path("/kaggle/working").exists():
    DATA_DIR = Path("/tmp/fongbe")
    WORK_DIR = Path("/kaggle/working/fongbe_asr")
elif Path("/content").exists():
    DATA_DIR = Path("/content/fongbe_data")
    WORK_DIR = Path("/content/fongbe_asr")
else:
    DATA_DIR = Path("fongbe_data")
    WORK_DIR = Path("fongbe_asr")

RUN_NAME = f"w2vbert_{NUM_LAYERS}L_clean"
OUT_DIR = WORK_DIR / "outputs" / RUN_NAME
WEIGHTS_DIR = WORK_DIR / "weights" / f"{RUN_NAME}_int8"
DATA_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_NAME={RUN_NAME} | DATA_DIR={DATA_DIR} | OUT_DIR={OUT_DIR}")

In [ ]:
# 1. Installer les bibliothèques (versions officielles + ONNX Runtime)
!pip install -q "transformers==5.14.1" soundfile "rapidfuzz==3.14.5" onnx onnxruntime

In [ ]:
# 2. Déployer le code du pipeline (100% autonome, sans Git)
from pathlib import Path

EMBEDDED_FILES = {'normalize.py': '"""Fon text normalisation used by the official score (standard library only).\n\nThree writing spaces, from richest to poorest:\n\n    raw ──normalize()─────► canonical form: NFD, lower case, look-alike characters fixed\n        ──tonal_target()──► what is pronounced: letters, spaces, tones; no punctuation, no digits\n        ──strip_tones()───► the same letters, with the tones removed from the vowels\n\nThe official score (`score.py`) works in the `tonal_target()` space, applied to the reference AND to the submission.\n`eval_space(t) = strip_tones(tonal_target(t))` is the tone-free space, given for diagnostics.\n\nDocumented choices:\n- NFD internally: `ɔ́` and `ɛ́` have no precomposed form; in NFC the corpus would be mixed.\n- Elision apostrophes are removed (`ɖ\'é` -> `ɖé`), so writing them or not makes no difference.\n- Digits are removed: the convention is to transcribe the spoken language; the few references that still\n  contain digits are kept as they are.\n- A tone is only removed by `strip_tones()` when it sits on a vowel: on a consonant (`ń`) it is part of the letter.\n"""\nfrom __future__ import annotations\n\nimport re\nimport unicodedata\n\n# Fon tones, as combining marks (NFD)\nACUTE, CARON, GRAVE, CIRCUMFLEX, MACRON = "́", "̌", "̀", "̂", "̄"\nTONES = frozenset({ACUTE, CARON, GRAVE, CIRCUMFLEX, MACRON})\n# Diacritics with no tonal value in Fon, found in the corpus: removed\nSPURIOUS = frozenset({"̧", "̈", "̣"})     # cedilla, diaeresis, dot below\nLETTERS = frozenset("abcdefghijklmnopqrstuvwxyz") | {"ɔ", "ɖ", "ɛ"}\nDIGITS = frozenset("0123456789")\nVOWELS = frozenset("aeiou") | {"ɔ", "ɛ"}\nMARKUP = frozenset("<>+*")\nLOOK_ALIKES = {                  # wrong character -> correct letter or sign\n    "ε": "ɛ", "Ε": "ɛ",                      # Greek epsilon\n    "ð": "ɖ", "đ": "ɖ", "ɗ": "ɖ",            # barred or hooked d\n    "‚": ",", "“": \'"\', "”": \'"\', "‘": "\'", "’": "\'",\n    "œ": "oe",\n}\n_SPACES = re.compile(r" {2,}")\n\n\ndef normalize(text: str) -> str:\n    """Canonical form: NFD, lower case, look-alike characters fixed. Idempotent."""\n    t = unicodedata.normalize("NFD", text).lower()\n    t = unicodedata.normalize("NFD", t)          # lower-casing can recompose characters\n    for wrong, right in LOOK_ALIKES.items():\n        t = t.replace(wrong, right)\n    return unicodedata.normalize("NFD", t)\n\n\ndef tonal_target(text: str, digits: bool = False) -> str:\n    """What is pronounced: letters, spaces and tones. Everything else is removed.\n    `digits=True` (training variant only): each digit 0-9 is kept as a separate word ("21h" -> "2 1 h");\n    `rejoin_digits` rebuilds the numbers after decoding. The official score always uses `digits=False`."""\n    out: list[str] = []\n    for c in normalize(text):\n        if c in TONES:\n            # a tone with no letter before it (typed after a space) is not pronounced\n            if out and out[-1] != " ":\n                out.append(c)\n        elif digits and c in DIGITS:\n            out.extend((" ", c, " "))           # one digit = one "word"\n        elif c in LETTERS or c == " ":\n            out.append(c)\n        elif c.isdigit() or c in SPURIOUS or c in MARKUP \\\n                or unicodedata.category(c).startswith("P"):\n            continue\n        elif c.isspace():\n            out.append(" ")\n        # any other character (outside the alphabet) is removed\n    return _SPACES.sub(" ", "".join(out)).strip()\n\n\ndef strip_tones(text: str) -> str:\n    """Remove the tones carried by a vowel; keep those carried by a consonant (they are letters)."""\n    out, base = [], ""\n    for c in unicodedata.normalize("NFD", text):\n        if c in TONES:\n            if base in VOWELS:\n                continue\n        else:\n            base = c\n        out.append(c)\n    return "".join(out)\n\n\ndef eval_space(text: str, digits: bool = False) -> str:\n    """Tone-free space (word error rate without tones)."""\n    return " ".join(strip_tones(tonal_target(text, digits)).split())\n\n\ndef eval_space_tonal(text: str, digits: bool = False) -> str:\n    """Tonal space (word error rate with tones)."""\n    return " ".join(tonal_target(text, digits).split())\n\n\ndef rejoin_digits(t: str) -> str:\n    """Output of a `digits=True` model: consecutive single digits become a number again ("2 1" -> "21")."""\n    out: list[str] = []\n    for w in t.split():\n        if w.isdigit() and out and out[-1].isdigit():\n            out[-1] += w\n        else:\n            out.append(w)\n    return " ".join(out)\n\n\ndef digit_sequence(t: str) -> str:\n    """Ordered sequence of the digits of a text, without separators (insensitive to grouping)."""\n    return "".join(c for c in unicodedata.normalize("NFD", t) if c in DIGITS)\n\n\ndef _self_test() -> None:\n    assert eval_space("Un nɔ́ ɖɔxó, kpódó mɛ!") == "un nɔ ɖɔxo kpodo mɛ"\n    assert eval_space("D\'é mɛ 2026 \\"bureau\\".") == "dé mɛ bureau".replace("é", "e")\n    assert eval_space_tonal("Kɔ́ 2 026 F, 3.5 %", digits=True) == unicodedata.normalize("NFD", "kɔ́ 2 0 2 6 f 3 5")\n    assert eval_space("ɖo 21h", digits=True) == "ɖo 2 1 h" and rejoin_digits("ɖo 2 1 h 0 5") == "ɖo 21 h 05"\n    assert digit_sequence("Numéro : 05 55, le 2 026") == "05552026"\n    assert eval_space("Kɔ 2026", digits=False) == "kɔ"\n    assert eval_space_tonal("Mǐ ná yi") == unicodedata.normalize("NFD", "mǐ ná yi")\n    assert eval_space("ɛ́ ε ð") == "ɛ ɛ ɖ"                  # look-alike characters\n    assert strip_tones(unicodedata.normalize("NFD", "ń")) == unicodedata.normalize("NFD", "ń")\n    assert eval_space("  a   ́ b ") == "a b"                 # orphan tone, multiple spaces\n    for t in ("Nɛ́ ká gbɔn", "É cí nú mì ɖɔ"):\n        assert normalize(normalize(t)) == normalize(t)     # idempotence\n    print("normalize.py: self-test OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'score.py': '#!/usr/bin/env python\n"""Official score of the "Frugal Tonal ASR for Fongbe" competition.\n\n    python score.py --reference Reference.csv --submission my_submission.csv [--usage Public|Private] [--output report.json]\n\nThe submission uses the Zindi format `ID, Target` (see SampleSubmission.csv); the columns `id, transcription`\nof the training files are also accepted, so that local validation files can be scored as they are.\nThe reference has `id, transcription` (or `ID, Target`), and optionally `Usage`.\nThe text is first normalised by `normalize.py`: Unicode NFD, lower case, punctuation, digits and elision\napostrophes removed. Letters and tones are kept.\n\nEach text is then split into syllables (onset + vowel + optional syllable-final "n", and its tone):\n  TSER (70 %): edit distance between the syllable sequences (base + tone) / number of reference syllables, capped at 1\n  conditional TER (30 %): among the syllables matched without looking at the tone, the share whose tone is wrong\n  score = 0.7 x TSER + 0.3 x conditional TER  (lower is better)\nErrors are accumulated over the whole set; an id missing from the submission counts as an empty transcription.\nDiagnostics (no effect on the ranking): segmental syllable error without tones, matched syllable rate, TER per tone.\n"""\nimport argparse\nimport collections\nimport csv\nimport json\nimport sys\nimport unicodedata as ud\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom normalize import tonal_target  # noqa: E402\n\nfrom rapidfuzz.distance import Levenshtein  # required, version pinned in requirements.txt\n\nVOWELS = set("aeiouɛɔə")\nTONES = {"́": "high", "̀": "low", "̌": "rising", "̂": "falling", "̄": "mid"}\n\n\ndef units(word):\n    """Base characters with their combining marks: [[char, [marks]], ...]."""\n    out = []\n    for c in word:\n        if ud.category(c).startswith("M") and out:\n            out[-1][1].append(c)\n        elif not ud.category(c).startswith("M"):\n            out.append([c, []])\n    return out\n\n\ndef syllables(text):\n    """List of (base, tone): base = onset + vowel (+ syllable-final n), tone = sorted tone marks (\'\' = unmarked)."""\n    res = []\n    for word in tonal_target(text or "").split():\n        u = units(word); i = 0; onset = ""\n        while i < len(u):\n            b, m = u[i]\n            tone = "".join(sorted(x for x in m if x in TONES)); other = "".join(x for x in m if x not in TONES)\n            if b in VOWELS:\n                base = onset + b + other; onset = ""\n                if i + 1 < len(u) and u[i + 1][0] == "n" and not u[i + 1][1] and (i + 2 >= len(u) or u[i + 2][0] not in VOWELS):\n                    base += "n"; i += 1\n                res.append((base, tone)); i += 1\n            elif tone:                                  # syllabic consonant carrying a tone (ń, m̀…)\n                res.append((onset + b + other, tone)); onset = ""; i += 1\n            else:\n                onset += b + other; i += 1\n        if onset:\n            res.append((onset, ""))\n    return res\n\n\ndef _distance(a, b):\n    return Levenshtein.distance(a, b)\n\n\ndef _equal_pairs(a, b):\n    """Pairs (i, j) matched identically in the rapidfuzz edit alignment."""\n    for op in Levenshtein.opcodes(a, b):\n        if op.tag == "equal":\n            for k in range(op.src_end - op.src_start):\n                yield op.src_start + k, op.dest_start + k\n\n\ndef measure(pairs):\n    """pairs: list of (reference, hypothesis). Errors are accumulated over the whole set."""\n    c = collections.Counter(); per_tone = collections.Counter(); per_tone_err = collections.Counter()\n    for ref, hyp in pairs:\n        r, h = syllables(ref), syllables(hyp)\n        c["ref_syllables"] += len(r)\n        c["tonal_distance"] += _distance([b + "|" + t for b, t in r], [b + "|" + t for b, t in h])\n        rb, hb = [b for b, _ in r], [b for b, _ in h]\n        c["segmental_distance"] += _distance(rb, hb)\n        for i, j in _equal_pairs(rb, hb):\n            tr, th = r[i][1], h[j][1]\n            c["matched"] += 1; c["wrong_tone"] += tr != th\n            cat = "+".join(TONES[x] for x in tr) or "unmarked"\n            per_tone[cat] += 1; per_tone_err[cat] += tr != th\n    tser = min(1.0, c["tonal_distance"] / max(1, c["ref_syllables"]))\n    terc = c["wrong_tone"] / max(1, c["matched"])\n    return {"score": 0.7 * tser + 0.3 * terc, "TSER": tser, "TER_conditional": terc,\n            "segmental_SER_without_tones": c["segmental_distance"] / max(1, c["ref_syllables"]),\n            "matched_syllable_rate": c["matched"] / max(1, c["ref_syllables"]),\n            "TER_per_tone": {k: round(per_tone_err[k] / per_tone[k], 4) for k in sorted(per_tone)},\n            "counts": {k: int(v) for k, v in c.items()}}\n\n\ndef read_csv(path):\n    with open(path, encoding="utf-8-sig", newline="") as f:\n        return list(csv.DictReader(f))\n\n\ndef id_and_text(row):\n    """(id, text) of a CSV row, in the Zindi format `ID, Target` or in the format `id, transcription`."""\n    if "ID" in row:\n        return row["ID"], row.get("Target") or ""\n    return row["id"], row.get("transcription") or ""\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__.split("\\n")[1])\n    ap.add_argument("--reference", required=True)\n    ap.add_argument("--submission", required=True)\n    ap.add_argument("--usage", choices=["Public", "Private"], help="score only one part of the test")\n    ap.add_argument("--output", help="JSON file for the detailed report")\n    a = ap.parse_args()\n    refs = [r for r in read_csv(a.reference) if not a.usage or r.get("Usage") == a.usage]\n    refs = [id_and_text(r) for r in refs]\n    sub = dict(id_and_text(r) for r in read_csv(a.submission))\n    missing = sum(1 for i, _ in refs if i not in sub)\n    report = measure([(text, sub.get(i, "")) for i, text in refs])\n    report.update({"utterances": len(refs), "missing_ids_scored_as_empty": missing, "usage": a.usage or "all"})\n    print(f"score {report[\'score\']:.4f}  TSER {report[\'TSER\']:.4f}  conditional TER {report[\'TER_conditional\']:.4f}  "\n          f"({len(refs)} utterances, {missing} missing)")\n    if a.output:\n        Path(a.output).write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', 'syllabify.py': '#!/usr/bin/env python\n"""Syllable pre-processing for the "Frugal Tonal ASR for Fongbe" leaderboard (standard library only).\n\nThe leaderboard metric is the usual word error rate (WER), computed on syllables instead of words.\nThe organisers apply `syllabify()` to the ground truth; participants MUST apply the same function to\ntheir transcriptions before submitting. The WER then counts one error for each wrong syllable:\na wrong tone makes one syllable wrong (not the whole word), and the way words are split no longer matters.\n\n    from syllabify import syllabify\n    syllabify("Un nɔ́ ɖu nǔ")          # -> \'un nɔ1 ɖu nu3\'\n\n    python syllabify.py my_transcriptions.csv my_submission.csv\n    # input: columns id, transcription (plain Fongbe text) or ID, Target\n    # output: the submission file, columns ID, Target (format of SampleSubmission.csv)\n\nWhat the function does:\n1. normalisation (same as normalize.py): Unicode NFD, lower case, look-alike characters fixed,\n   punctuation, digits and elision apostrophes removed; letters and tones kept;\n2. each word is cut into syllables: onset consonants + vowel (+ syllable-final n), or a syllabic consonant\n   carrying a tone (ń, m̀...);\n3. the tone of each syllable is written as a digit after it, so that it survives any further text cleaning:\n   1 = high (acute), 2 = low (grave), 3 = rising (caron), 4 = falling (circumflex), 5 = mid (macron);\n   no digit = no tone mark;\n4. the syllables are joined with single spaces.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport re\nimport sys\nimport unicodedata\n\n# ---- normalisation: identical to normalize.tonal_target(text, digits=False) ----\nACUTE, CARON, GRAVE, CIRCUMFLEX, MACRON = "́", "̌", "̀", "̂", "̄"\nTONES = frozenset({ACUTE, CARON, GRAVE, CIRCUMFLEX, MACRON})\nSPURIOUS = frozenset({"̧", "̈", "̣"})     # cedilla, diaeresis, dot below\nLETTERS = frozenset("abcdefghijklmnopqrstuvwxyz") | {"ɔ", "ɖ", "ɛ"}\nMARKUP = frozenset("<>+*")\nLOOK_ALIKES = {\n    "ε": "ɛ", "Ε": "ɛ", "ð": "ɖ", "đ": "ɖ", "ɗ": "ɖ",\n    "‚": ",", "“": \'"\', "”": \'"\', "‘": "\'", "’": "\'", "œ": "oe",\n}\n_SPACES = re.compile(r" {2,}")\n\n\ndef normalize(text: str) -> str:\n    t = unicodedata.normalize("NFD", text).lower()\n    t = unicodedata.normalize("NFD", t)\n    for wrong, right in LOOK_ALIKES.items():\n        t = t.replace(wrong, right)\n    return unicodedata.normalize("NFD", t)\n\n\ndef tonal_target(text: str) -> str:\n    out: list[str] = []\n    for c in normalize(text):\n        if c in TONES:\n            if out and out[-1] != " ":\n                out.append(c)\n        elif c in LETTERS or c == " ":\n            out.append(c)\n        elif c.isdigit() or c in SPURIOUS or c in MARKUP or unicodedata.category(c).startswith("P"):\n            continue\n        elif c.isspace():\n            out.append(" ")\n    return _SPACES.sub(" ", "".join(out)).strip()\n\n\n# ---- syllables: identical to score.syllables() ----\nVOWELS = set("aeiouɛɔə")\nTONE_DIGIT = {ACUTE: "1", GRAVE: "2", CARON: "3", CIRCUMFLEX: "4", MACRON: "5"}\n\n\ndef _units(word):\n    out = []\n    for c in word:\n        if unicodedata.category(c).startswith("M") and out:\n            out[-1][1].append(c)\n        elif not unicodedata.category(c).startswith("M"):\n            out.append([c, []])\n    return out\n\n\ndef syllables(text: str) -> list[tuple[str, str]]:\n    """List of (base, tone marks) for a text."""\n    res = []\n    for word in tonal_target(text or "").split():\n        u = _units(word); i = 0; onset = ""\n        while i < len(u):\n            b, m = u[i]\n            tone = "".join(sorted(x for x in m if x in TONES)); other = "".join(x for x in m if x not in TONES)\n            if b in VOWELS:\n                base = onset + b + other; onset = ""\n                if i + 1 < len(u) and u[i + 1][0] == "n" and not u[i + 1][1] and (i + 2 >= len(u) or u[i + 2][0] not in VOWELS):\n                    base += "n"; i += 1\n                res.append((base, tone)); i += 1\n            elif tone:\n                res.append((onset + b + other, tone)); onset = ""; i += 1\n            else:\n                onset += b + other; i += 1\n        if onset:\n            res.append((onset, ""))\n    return res\n\n\ndef syllabify(text: str) -> str:\n    """Text -> space-separated syllables with tone digits, e.g. \'Un nɔ́ ɖu nǔ\' -> \'un nɔ1 ɖu nu3\'."""\n    return " ".join(unicodedata.normalize("NFC", base) + "".join(TONE_DIGIT[t] for t in tone)\n                    for base, tone in syllables(text))\n\n\ndef _self_test() -> None:\n    assert syllabify("Un nɔ́ ɖu nǔ") == "un nɔ1 ɖu nu3"\n    assert syllabify("Kɔ́xojí, ɖ\'é mɛ !") == syllabify("kɔ́ xo jí ɖé mɛ") == "kɔ1 xo ji1 ɖe1 mɛ"\n    assert syllabify("Azɔ̌mɛ 2026") == "a zɔ3 mɛ"\n    assert syllabify("") == "" and syllabify("?!") == ""\n    s = syllabify("Mǐ ná yi gbɛ̀ ń")\n    assert s == "mi3 na1 yi gbɛ2 n1", s\n    print("syllabify.py: self-test OK")\n\n\ndef _convert(src: str, dst: str) -> None:\n    with open(src, encoding="utf-8-sig", newline="") as f:\n        rows = list(csv.DictReader(f))\n    with open(dst, "w", encoding="utf-8", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["ID", "Target"])\n        for r in rows:\n            if "ID" in r:\n                w.writerow([r["ID"], syllabify(r.get("Target") or "")])\n            else:\n                w.writerow([r["id"], syllabify(r.get("transcription") or "")])\n    print(f"{len(rows)} rows written to {dst}")\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 3:\n        _convert(sys.argv[1], sys.argv[2])\n    else:\n        _self_test()\n', 'measure_frugality.py': '#!/usr/bin/env python\n"""Frugality measurement of the "Frugal Tonal ASR for Fongbe" competition: the script the organisers run on the\ntop 20 of the private leaderboard, provided so that every team can check its compliance before submitting.\n\n    python measure_frugality.py \\\n        --weights my_system/ --weights lm.bin \\\n        --command "python infer.py --csv {csv} --root {root} --output {output}" \\\n        --csv Test.csv [--limit 300] [--report report.json]\n\nYour command reads a CSV (`id, audio_path`, paths relative to `{root}`) and writes `{output}` (`id, transcription`,\nor the submission format `ID, Target`).\nWhat the script measures, on the complete system, from audio to tonal transcription:\n\n  size        sum of the files given with --weights: ALL the weights (pre-trained models and language model included,\n              after any quantisation), threshold 1,000 MB (MB = 10^6 bytes)\n  RTF         elapsed time of the command / total audio duration, on ONE single CPU core, threshold 1\n  memory      peak resident memory of the command and all its sub-processes, threshold 2 GB (GB = 10^9 bytes)\n  offline     the command runs with the network cut (Linux, root rights: `unshare -n`) and without any GPU\n\nIf the CSV has a `transcription` column (for example the `recommended_validation` voices of Train_reliable.csv),\nthe official score is recomputed with `score.py`. The organisers recompute it on the full test set.\n\nConditions of the official check: recent x86-64 Linux machine, a single thread (CPU affinity + *_NUM_THREADS variables),\nno GPU, network cut; model loading counts in the measured time. The thresholds are those of the competition rules;\nexceeding one excludes a team from the prizes, it does not change the leaderboard.\n"""\nimport argparse\nimport csv\nimport json\nimport os\nimport resource\nimport shlex\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport wave\nfrom pathlib import Path\n\nMAX_SIZE = 1000 * 10 ** 6       # 1,000 MB\nMAX_RTF = 1.0\nMAX_MEMORY = 2 * 10 ** 9        # 2 GB\nTHREADS = {k: "1" for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",\n                            "VECLIB_MAXIMUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS", "ONNXRUNTIME_NUM_THREADS")}\nOFFLINE = {"CUDA_VISIBLE_DEVICES": "", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"}\nNETWORK_NOT_CUT = "network NOT cut"\n\n\ndef weights_size(paths):\n    seen, total, detail = set(), 0, {}\n    for c in paths:\n        p = Path(c)\n        files = [p] if p.is_file() else [f for f in p.rglob("*") if f.is_file()] if p.is_dir() else []\n        if not files:\n            sys.exit(f"--weights not found or empty: {c}")\n        t = 0\n        for f in files:\n            key = f.resolve()\n            if key not in seen:                    # the same file (link) is only counted once\n                seen.add(key); t += f.stat().st_size\n        detail[str(c)] = t; total += t\n    return total, detail\n\n\ndef wav_duration(path):\n    with wave.open(str(path)) as w:\n        return w.getnframes() / w.getframerate()\n\n\ndef tree_memory(pid):\n    """Resident memory (bytes) of the process and all its descendants, read from /proc (Linux)."""\n    total, todo = 0, [pid]\n    while todo:\n        p = todo.pop()\n        try:\n            for line in open(f"/proc/{p}/status"):\n                if line.startswith("VmRSS:"):\n                    total += int(line.split()[1]) * 1024\n                    break\n            for task in os.listdir(f"/proc/{p}/task"):\n                with open(f"/proc/{p}/task/{task}/children") as f:\n                    todo.extend(int(x) for x in f.read().split())\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            continue\n    return total\n\n\ndef run(command, env, one_core, no_network):\n    prefix, notes = [], []\n    if no_network:\n        if shutil.which("unshare") and os.geteuid() == 0:\n            prefix = ["unshare", "-n"]\n        else:\n            notes.append(f"{NETWORK_NOT_CUT} (Linux, `unshare` and root rights are needed): check yourself that no network call is made")\n    core = None\n    if one_core and hasattr(os, "sched_setaffinity"):\n        core = min(os.sched_getaffinity(0))\n    elif one_core:\n        notes.append("single core NOT enforced (Linux needed): only the *_NUM_THREADS variables limit the threads")\n\n    def before():\n        if core is not None:\n            os.sched_setaffinity(0, {core})\n\n    t0 = time.perf_counter()\n    proc = subprocess.Popen(prefix + ["/bin/sh", "-c", command], env=env, preexec_fn=before if os.name == "posix" else None)\n    peak = 0\n    while proc.poll() is None:\n        if os.path.isdir("/proc"):\n            peak = max(peak, tree_memory(proc.pid))\n        time.sleep(0.05)\n    elapsed = time.perf_counter() - t0\n    # complement: the largest descendant process seen by the kernel (ru_maxrss in KiB on Linux)\n    peak = max(peak, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss * 1024)\n    return proc.returncode, elapsed, peak, notes, core\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="Frugality measurement (size, RTF on one core, peak memory, offline).")\n    ap.add_argument("--weights", action="append", required=True, help="weights folder or file; repeat for each one")\n    ap.add_argument("--command", required=True, help="inference command with {csv}, {root} and {output}")\n    ap.add_argument("--csv", required=True, help="CSV with id, audio_path (and transcription to recompute the score)")\n    ap.add_argument("--root", help="folder the audio_path paths are relative to (default: the folder of the CSV)")\n    ap.add_argument("--limit", type=int, help="only measure the first N utterances (quick check)")\n    ap.add_argument("--report", help="JSON file for the report")\n    ap.add_argument("--allow-network", action="store_true", help="do not cut the network (not recommended)")\n    a = ap.parse_args()\n\n    source = Path(a.csv).resolve()\n    root = Path(a.root).resolve() if a.root else source.parent\n    with open(source, encoding="utf-8-sig", newline="") as f:\n        rows = list(csv.DictReader(f))\n    if a.limit:\n        rows = rows[:a.limit]\n    if not rows or not {"id", "audio_path"} <= set(rows[0]):\n        sys.exit("the CSV needs at least the columns id and audio_path")\n    audio_total = sum(wav_duration(root / r["audio_path"]) for r in rows)\n\n    size, detail = weights_size(a.weights)\n    work = Path(tempfile.mkdtemp(prefix="frugality-"))\n    input_csv, output = work / "input.csv", work / "submission.csv"\n    with open(input_csv, "w", encoding="utf-8", newline="") as f:\n        w = csv.writer(f); w.writerow(["id", "audio_path"]); w.writerows((r["id"], r["audio_path"]) for r in rows)\n    command = a.command.format(csv=shlex.quote(str(input_csv)), root=shlex.quote(str(root)), output=shlex.quote(str(output)))\n    env = dict(os.environ, **THREADS, **OFFLINE)\n    print(f"{len(rows)} utterances, {audio_total / 60:.1f} min of audio; weights: {size / 1e6:.1f} MB; running…", flush=True)\n    code, elapsed, peak, notes, core = run(command, env, True, not a.allow_network)\n\n    report = {"utterances": len(rows), "audio_s": round(audio_total, 2), "command": a.command, "return_code": code,\n              "size_bytes": size, "size_detail": detail, "time_s": round(elapsed, 2),\n              "RTF": round(elapsed / audio_total, 4) if audio_total else None, "peak_memory_bytes": peak,\n              "core": core, "threads": THREADS, "notes": notes}\n    if code != 0 or not output.exists():\n        report["error"] = f"the command failed (code {code}) or did not write {output}"\n    else:\n        with open(output, encoding="utf-8-sig", newline="") as f:\n            sub = {(r["ID"] if "ID" in r else r["id"]): (r.get("Target") if "ID" in r else r.get("transcription")) or ""\n                   for r in csv.DictReader(f)}\n        ids = [r["id"] for r in rows]\n        report["submission"] = {"missing": sum(i not in sub for i in ids), "extra": len(set(sub) - set(ids)),\n                                "empty": sum(not sub.get(i, "").strip() for i in ids)}\n        if "transcription" in rows[0]:\n            sys.path.insert(0, str(Path(__file__).resolve().parent))\n            from score import measure\n            s = measure([(r["transcription"], sub.get(r["id"], "")) for r in rows])\n            report["score"] = {k: s[k] for k in ("score", "TSER", "TER_conditional")}\n\n    def verdict(value, limit):\n        return "not measured" if value is None else ("compliant" if value <= limit else "NOT COMPLIANT")\n    report["verdict"] = {"size": verdict(size, MAX_SIZE), "RTF": verdict(report["RTF"], MAX_RTF),\n                         "memory": verdict(peak, MAX_MEMORY),\n                         "offline": "not checked" if any(NETWORK_NOT_CUT in n for n in notes) else "network cut"}\n    print(f"size     {size / 1e6:9.1f} MB   (<= {MAX_SIZE / 1e6:.0f})   {report[\'verdict\'][\'size\']}")\n    print(f"RTF      {report[\'RTF\']:9.3f}      (<= {MAX_RTF:g})       {report[\'verdict\'][\'RTF\']}   [{elapsed:.1f} s for {audio_total:.1f} s of audio]")\n    print(f"memory   {peak / 1e9:9.3f} GB   (<= {MAX_MEMORY / 1e9:.0f})       {report[\'verdict\'][\'memory\']}")\n    print(f"network  {report[\'verdict\'][\'offline\']}")\n    if "score" in report:\n        print(f"score    {report[\'score\'][\'score\']:.4f}  (TSER {report[\'score\'][\'TSER\']:.4f}, conditional TER {report[\'score\'][\'TER_conditional\']:.4f})")\n    for n in notes + ([report["error"]] if "error" in report else []):\n        print("WARNING:", n)\n    if a.report:\n        Path(a.report).write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")\n    shutil.rmtree(work, ignore_errors=True)\n    sys.exit(0 if "error" not in report and all(v in ("compliant", "network cut") for v in report["verdict"].values()) else 1)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/__init__.py': '"""Frugal Tonal ASR for Fongbe — Personne 2 (ML / ASR) pipeline."""\n', 'src/features_numpy.py': '"""Pure-NumPy Kaldi log-Mel + stride-2 feature extractor for facebook/w2v-bert-2.0.\n\nZero PyTorch / Transformers dependency at inference time (saves ~400-500 MB RAM\nand several seconds of startup time under `measure_frugality.py`).\n\nMatches Hugging Face `SeamlessM4TFeatureExtractor` (80 Kaldi mel bins, 16 kHz,\n25 ms frame / 10 ms shift, Povey window, preemphasis 0.97, utterance CMVN,\nstride-2 stacking -> 160-dim features). Parity is checked automatically by\n`src/export_onnx_int8.py`.\n"""\nfrom __future__ import annotations\n\nimport math\nimport numpy as np\n\n\ndef _mel_scale(freq: np.ndarray) -> np.ndarray:\n    return 1127.0 * np.log(1.0 + freq / 700.0)\n\n\ndef _kaldi_mel_filterbank(\n    num_mel_bins: int = 80,\n    n_fft: int = 512,\n    sample_rate: int = 16000,\n    low_freq: float = 20.0,\n    high_freq: float = 8000.0,\n) -> np.ndarray:\n    """Kaldi triangular mel filterbank matrix of shape (n_fft // 2 + 1, num_mel_bins)."""\n    num_fft_bins = n_fft // 2\n    fft_bin_width = sample_rate / n_fft\n    mel_low = float(_mel_scale(np.array(low_freq)))\n    mel_high = float(_mel_scale(np.array(high_freq)))\n    mel_delta = (mel_high - mel_low) / (num_mel_bins + 1)\n\n    bins = np.arange(num_mel_bins, dtype=np.float64)\n    left_mel = mel_low + bins * mel_delta\n    center_mel = mel_low + (bins + 1.0) * mel_delta\n    right_mel = mel_low + (bins + 2.0) * mel_delta\n\n    freqs = np.arange(num_fft_bins, dtype=np.float64) * fft_bin_width\n    mel = _mel_scale(freqs)[:, None]  # (num_fft_bins, 1)\n\n    up_slope = (mel - left_mel[None, :]) / (center_mel[None, :] - left_mel[None, :])\n    down_slope = (right_mel[None, :] - mel) / (right_mel[None, :] - center_mel[None, :])\n    fb = np.maximum(0.0, np.minimum(up_slope, down_slope))\n\n    # Append Nyquist bin (zeroed, matching Kaldi / HF audio_utils)\n    fb = np.vstack([fb, np.zeros((1, num_mel_bins), dtype=np.float64)])\n    return fb.astype(np.float32)\n\n\n_MEL_FB_80 = _kaldi_mel_filterbank(num_mel_bins=80, n_fft=512, sample_rate=16000, low_freq=20.0, high_freq=8000.0)\n_POVEY_400 = np.power(0.5 - 0.5 * np.cos(2.0 * math.pi * np.arange(400, dtype=np.float64) / 399.0), 0.85).astype(\n    np.float32\n)\n\n\ndef extract_w2vbert_features(\n    waveform: np.ndarray,\n    sample_rate: int = 16000,\n    stride: int = 2,\n    padding_value: float = 1.0,\n) -> np.ndarray:\n    """Extract (T_out, 160) float32 features matching SeamlessM4TFeatureExtractor.\n\n    Args:\n        waveform: 1D float32 audio in [-1, 1] at 16 kHz.\n        sample_rate: Must be 16000.\n        stride: Frame stacking stride (2 for w2v-bert-2.0 -> 80 * 2 = 160 dims).\n        padding_value: Pad value if odd frame count (unused when dropping/zero-padding).\n    """\n    if sample_rate != 16000:\n        raise ValueError(f"Expected 16000 Hz audio, got {sample_rate}")\n    wav = np.asarray(waveform, dtype=np.float32)\n    if wav.ndim > 1:\n        wav = wav.mean(axis=-1)\n\n    # SeamlessM4TFeatureExtractor multiplies waveform by 2**15 (Kaldi int16 scale)\n    wav = wav * 32768.0\n\n    frame_length = 400  # 25 ms\n    frame_shift = 160   # 10 ms\n    n_fft = 512\n\n    if len(wav) < frame_length:\n        wav = np.pad(wav, (0, frame_length - len(wav)), mode="constant")\n\n    num_frames = 1 + (len(wav) - frame_length) // frame_shift\n    # Strided framing view (num_frames, 400)\n    itemsize = wav.strides[0]\n    frames = np.lib.stride_tricks.as_strided(\n        wav,\n        shape=(num_frames, frame_length),\n        strides=(frame_shift * itemsize, itemsize),\n    ).copy()\n\n    # Remove DC offset per frame\n    frames -= frames.mean(axis=1, keepdims=True)\n\n    # Pre-emphasis (0.97)\n    preemph = 0.97\n    frames[:, 1:] = frames[:, 1:] - preemph * frames[:, :-1]\n    frames[:, 0] = frames[:, 0] * (1.0 - preemph)\n\n    # Povey window + 512-pt rFFT power spectrum\n    frames *= _POVEY_400[None, :]\n    spec = np.fft.rfft(frames, n=n_fft, axis=1)\n    power = (spec.real ** 2 + spec.imag ** 2).astype(np.float32)\n\n    # Mel filterbank + log\n    mel_energies = np.matmul(power, _MEL_FB_80)\n    log_mel = np.log(np.maximum(mel_energies, np.float32(1.1920929e-7)))\n\n    # Per-utterance zero-mean unit-variance normalisation (ddof=1 matches HF SeamlessM4T)\n    mean = log_mel.mean(axis=0, keepdims=True)\n    var = log_mel.var(axis=0, ddof=1 if len(log_mel) > 1 else 0, keepdims=True)\n    log_mel = (log_mel - mean) / np.sqrt(var + np.float32(1e-7))\n\n    # Stride-2 stacking (SeamlessM4T trims trailing odd frame: `num_frames // stride * stride`)\n    n_stacked = (log_mel.shape[0] // stride) * stride\n    if n_stacked == 0:\n        pad_n = stride - log_mel.shape[0]\n        log_mel = np.pad(log_mel, ((0, pad_n), (0, 0)), constant_values=padding_value)\n        n_stacked = stride\n    stacked = log_mel[:n_stacked].reshape(n_stacked // stride, log_mel.shape[1] * stride)\n    return np.ascontiguousarray(stacked, dtype=np.float32)\n', 'src/make_splits.py': '#!/usr/bin/env python\n"""Generate leak-free, test-mirroring train/validation splits and CTC vocab.\n\nTest set composition (from official README.md):\n  - General speech   : 6,382 utts (67.8%), 60 unseen voices\n  - Religious speech : 3,025 utts (32.2%), same reader (BIBLE_L01), held-out Bible books\n\nThis script mirrors that exact structure locally:\n  - val_general.csv   : recommended_validation == 1 from Train_reliable.csv (2,396 utts, 14 unseen voices)\n  - val_religious.csv : one held-out Bible book from Train_religious.csv (default: JDG, 1,194 utts, 2.81 h)\n  - val_mirror.csv    : combined validation set (~66.7% general / ~33.3% religious) with `domain` column\n  - train_clean.csv   : Train_reliable (rec_val=0) + remaining 8 books of Train_religious (~138 h)\n  - vocab.json        : full NFD tonal CTC vocabulary (29 base letters + 5 tones + | + [UNK] + [PAD])\n\nUsage:\n    python src/make_splits.py --data-root . --out-dir splits --val-book JDG\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nfrom normalize import LETTERS, TONES, tonal_target  # noqa: E402\n\n\ndef read_csv(path: Path) -> list[dict[str, str]]:\n    with open(path, encoding="utf-8-sig", newline="") as f:\n        return list(csv.DictReader(f))\n\n\ndef write_csv(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=fieldnames)\n        w.writeheader()\n        w.writerows(rows)\n\n\ndef make_row(r: dict[str, str], source: str, domain: str) -> dict[str, object]:\n    raw = r.get("transcription") or ""\n    tgt = tonal_target(raw, digits=False)\n    return {\n        "id": r["id"],\n        "audio_path": r["audio_path"],\n        "transcription": raw,\n        "target": tgt,\n        "speaker_id": r.get("speaker_id", ""),\n        "duration": float(r["duration"]),\n        "source": source,\n        "domain": domain,\n        "meta": r.get("book", "") or r.get("protocol", ""),\n    }\n\n\ndef build_vocab() -> dict[str, int]:\n    """Deterministic NFD character vocab covering all official letters and tones."""\n    chars = sorted(LETTERS) + sorted(TONES)\n    vocab = {c: i for i, c in enumerate(chars)}\n    vocab["|"] = len(vocab)\n    vocab["[UNK]"] = len(vocab)\n    vocab["[PAD]"] = len(vocab)\n    return vocab\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Create leak-free mirror splits and CTC vocab.")\n    ap.add_argument("--data-root", type=Path, default=ROOT, help="Folder containing Train_*.csv")\n    ap.add_argument("--out-dir", type=Path, default=ROOT / "splits", help="Output folder for split CSVs")\n    ap.add_argument("--val-book", default="JDG", help="Held-out Bible book code for val_religious (default: JDG)")\n    a = ap.parse_args()\n\n    reliable_raw = read_csv(a.data_root / "Train_reliable.csv")\n    religious_raw = read_csv(a.data_root / "Train_religious.csv")\n\n    train_rel = [make_row(r, "reliable", "general") for r in reliable_raw if r["recommended_validation"] == "0"]\n    val_gen = [make_row(r, "reliable", "general") for r in reliable_raw if r["recommended_validation"] == "1"]\n\n    available_books = sorted({r["book"] for r in religious_raw})\n    if a.val_book not in available_books:\n        sys.exit(f"--val-book {a.val_book!r} not found in Train_religious.csv. Available: {available_books}")\n\n    train_bib = [make_row(r, "religious", "religious") for r in religious_raw if r["book"] != a.val_book]\n    val_bib = [make_row(r, "religious", "religious") for r in religious_raw if r["book"] == a.val_book]\n\n    # Drop any empty normalized targets from training only (never drop from validation)\n    train_clean = [r for r in (train_rel + train_bib) if r["target"]]\n    val_mirror = val_gen + val_bib\n\n    # Integrity checks: zero speaker leakage on general, zero book leakage on religious\n    trn_spks = {r["speaker_id"] for r in train_rel}\n    val_spks = {r["speaker_id"] for r in val_gen}\n    assert not (trn_spks & val_spks), f"Speaker leakage detected: {trn_spks & val_spks}"\n    assert {r["meta"] for r in train_bib}.isdisjoint({r["meta"] for r in val_bib}), "Bible book leakage!"\n\n    fields = ["id", "audio_path", "transcription", "target", "speaker_id", "duration", "source", "domain", "meta"]\n    write_csv(a.out_dir / "train_clean.csv", train_clean, fields)\n    write_csv(a.out_dir / "train_reliable_only.csv", [r for r in train_rel if r["target"]], fields)\n    write_csv(a.out_dir / "val_general.csv", val_gen, fields)\n    write_csv(a.out_dir / "val_religious.csv", val_bib, fields)\n    write_csv(a.out_dir / "val_mirror.csv", val_mirror, fields)\n\n    vocab = build_vocab()\n    (a.out_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=2), encoding="utf-8")\n\n    def stats(name: str, rows: list[dict[str, object]]) -> str:\n        hrs = sum(float(r["duration"]) for r in rows) / 3600.0\n        spks = len({r["speaker_id"] for r in rows})\n        return f"  {name:22s}: {len(rows):6,d} utts | {hrs:6.2f} h | {spks:4d} speakers"\n\n    print(f"Splits written to {a.out_dir} (val_book={a.val_book}):")\n    print(stats("train_clean", train_clean))\n    print(stats("  - reliable (train)", [r for r in train_rel if r["target"]]))\n    print(stats("  - religious (train)", [r for r in train_bib if r["target"]]))\n    print(stats("val_general", val_gen))\n    print(stats("val_religious", val_bib))\n    print(stats("val_mirror", val_mirror))\n    gen_pct = 100.0 * len(val_gen) / len(val_mirror)\n    rel_pct = 100.0 * len(val_bib) / len(val_mirror)\n    print(f"  Mirror ratio          : {gen_pct:.1f}% general / {rel_pct:.1f}% religious (Test = 67.8% / 32.2%)")\n    print(f"  vocab.json            : {len(vocab)} tokens")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/evaluate.py': '#!/usr/bin/env python\n"""Local evaluation mirroring the Test set composition (67.8% general + 32.2% religious).\n\nComputes official `score.py` metrics (`TSER`, `segmental_SER_without_tones`, `TER_conditional`, `TER_per_tone`)\nseparately on `general` and `religious`, plus the weighted mirror TSER matching Test.csv proportions.\n\nUsage:\n    python src/evaluate.py --reference splits/val_mirror.csv --predictions outputs/val_pred.csv [--output report.json] [--show-errors 10]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\n\ntry:\n    from score import id_and_text, measure, syllables  # noqa: E402\nexcept ImportError as exc:\n    sys.exit(f"Missing dependency for official score.py ({exc}). Install with: pip install rapidfuzz==3.14.5")\n\n# Official Test.csv split from README.md: 6,382 general / 3,025 religious (total 9,407)\nTEST_WEIGHT_GENERAL = 6382 / 9407\nTEST_WEIGHT_RELIGIOUS = 3025 / 9407\n_DIGITS_RE = re.compile(r"[1-5]")\n\n\ndef read_csv(path: Path) -> list[dict[str, str]]:\n    with open(path, encoding="utf-8-sig", newline="") as f:\n        return list(csv.DictReader(f))\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Evaluate plain Fongbe predictions on mirror validation set.")\n    ap.add_argument("--reference", type=Path, required=True, help="Reference CSV (e.g. splits/val_mirror.csv)")\n    ap.add_argument("--predictions", type=Path, required=True, help="Predictions CSV (id,transcription)")\n    ap.add_argument("--output", type=Path, help="Optional JSON report path")\n    ap.add_argument("--show-errors", type=int, default=0, help="Print N worst utterances for P3 tonal analysis")\n    a = ap.parse_args()\n\n    refs = read_csv(a.reference)\n    preds = dict(id_and_text(r) for r in read_csv(a.predictions))\n\n    # Guard against accidentally scoring already-syllabified predictions (digits would be stripped!)\n    sample_hyps = [preds.get(id_and_text(r)[0], "") for r in refs[:50]]\n    digit_hits = sum(1 for h in sample_hyps if _DIGITS_RE.search(h))\n    if digit_hits > 10:\n        print(\n            "WARNING: Predictions appear to contain tone digits (1-5). "\n            "score.py expects PLAIN Fongbe text with diacritics, NOT syllabify() output!",\n            file=sys.stderr,\n        )\n\n    all_pairs = [(id_and_text(r)[1], preds.get(id_and_text(r)[0], "")) for r in refs]\n    missing = sum(1 for r in refs if id_and_text(r)[0] not in preds)\n    overall = measure(all_pairs)\n\n    report: dict[str, object] = {\n        "utterances": len(refs),\n        "missing": missing,\n        "overall_pooled": overall,\n    }\n\n    has_domain = bool(refs and "domain" in refs[0] and {r["domain"] for r in refs} >= {"general", "religious"})\n    if has_domain:\n        gen_pairs = [(id_and_text(r)[1], preds.get(id_and_text(r)[0], "")) for r in refs if r["domain"] == "general"]\n        rel_pairs = [(id_and_text(r)[1], preds.get(id_and_text(r)[0], "")) for r in refs if r["domain"] == "religious"]\n        m_gen = measure(gen_pairs)\n        m_rel = measure(rel_pairs)\n        mirror_tser = TEST_WEIGHT_GENERAL * m_gen["TSER"] + TEST_WEIGHT_RELIGIOUS * m_rel["TSER"]\n        report["general"] = m_gen\n        report["religious"] = m_rel\n        report["TSER_mirror"] = round(mirror_tser, 6)\n\n        print("=" * 72)\n        print(f"  TSER_mirror (67.8% gen + 32.2% rel) : {mirror_tser * 100:6.2f} %")\n        print("-" * 72)\n        print(\n            f"  general   ({len(gen_pairs):4d} utts) : TSER {m_gen[\'TSER\'] * 100:6.2f}% | "\n            f"seg_SER {m_gen[\'segmental_SER_without_tones\'] * 100:6.2f}% | "\n            f"cond_TER {m_gen[\'TER_conditional\'] * 100:6.2f}%"\n        )\n        print(\n            f"  religious ({len(rel_pairs):4d} utts) : TSER {m_rel[\'TSER\'] * 100:6.2f}% | "\n            f"seg_SER {m_rel[\'segmental_SER_without_tones\'] * 100:6.2f}% | "\n            f"cond_TER {m_rel[\'TER_conditional\'] * 100:6.2f}%"\n        )\n        print(f"  TER_per_tone (general)  : {m_gen[\'TER_per_tone\']}")\n        print(f"  TER_per_tone (religious): {m_rel[\'TER_per_tone\']}")\n        print("=" * 72)\n    else:\n        print(\n            f"TSER {overall[\'TSER\'] * 100:.2f}% | seg_SER {overall[\'segmental_SER_without_tones\'] * 100:.2f}% | "\n            f"cond_TER {overall[\'TER_conditional\'] * 100:.2f}% | missing={missing}"\n        )\n        print("TER_per_tone:", overall["TER_per_tone"])\n\n    if a.show_errors > 0:\n        from rapidfuzz.distance import Levenshtein\n\n        scored = []\n        for r in refs:\n            uid, ref_txt = id_and_text(r)\n            hyp_txt = preds.get(uid, "")\n            rs = [b + "|" + t for b, t in syllables(ref_txt)]\n            hs = [b + "|" + t for b, t in syllables(hyp_txt)]\n            err = Levenshtein.distance(rs, hs) / max(1, len(rs))\n            scored.append((err, uid, r.get("domain", ""), ref_txt, hyp_txt))\n        scored.sort(reverse=True)\n        print(f"\\nTop {a.show_errors} worst utterances:")\n        for err, uid, dom, ref_txt, hyp_txt in scored[: a.show_errors]:\n            print(f"  [{uid} | {dom} | TSER={err * 100:.1f}%]\\n    REF: {ref_txt}\\n    HYP: {hyp_txt}")\n\n    if a.output:\n        a.output.parent.mkdir(parents=True, exist_ok=True)\n        a.output.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n        print(f"Report written to {a.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/train_ctc.py': '#!/usr/bin/env python\n"""Train or fine-tune a CTC Tonal ASR model (facebook/w2v-bert-2.0, MIT licence).\n\nSupports:\n  - Full 24-layer Teacher OR layer-pruned frugal Student (`--num-layers 12` / `16` / `24`)\n  - Multi-domain training (`train_clean.csv` = reliable + religious, plus optional `--extra-train-csv`\n    for filtered/pseudo-labeled Train_uncertain)\n  - Automatic filtering to WAV files actually extracted on disk under `--data-dir`\n  - Tonal-safe SpecAugment (`--spec-augment`, never shifts pitch!)\n  - Resuming across Kaggle/Colab sessions (`--resume-from`)\n  - Mirror validation (`val_general` + `val_religious` -> `TSER_mirror`) saving the best checkpoint\n\nUsage (Kaggle / Colab / Linux GPU):\n    python src/train_ctc.py \\\n        --data-dir /tmp/fongbe \\\n        --train-csv splits/train_clean.csv \\\n        --val-csv splits/val_mirror.csv \\\n        --vocab splits/vocab.json \\\n        --out-dir outputs/w2vbert_12L \\\n        --num-layers 12 \\\n        --max-steps 6000 \\\n        --eval-steps 500\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import (\n    SeamlessM4TFeatureExtractor,\n    Wav2Vec2BertForCTC,\n    Wav2Vec2BertProcessor,\n    Wav2Vec2CTCTokenizer,\n    get_cosine_schedule_with_warmup,\n)\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nfrom normalize import tonal_target  # noqa: E402\nfrom score import measure  # noqa: E402\n\nTEST_WEIGHT_GENERAL = 6382 / 9407\nTEST_WEIGHT_RELIGIOUS = 3025 / 9407\n\n\ndef read_rows(path: Path, data_dir: Path, max_seconds: float | None = None, require_target: bool = True) -> list[dict]:\n    with open(path, encoding="utf-8-sig", newline="") as f:\n        raw_rows = list(csv.DictReader(f))\n    out = []\n    for r in raw_rows:\n        wav = data_dir / r["audio_path"]\n        if not wav.exists():\n            continue\n        dur = float(r.get("duration") or 0.0)\n        if max_seconds is not None and dur > max_seconds:\n            continue\n        tgt = (r.get("target") or tonal_target(r.get("transcription") or "")).strip()\n        if require_target and not tgt:\n            continue\n        out.append({\n            "id": r["id"],\n            "wav": str(wav),\n            "transcription": r.get("transcription") or "",\n            "target": tgt,\n            "duration": dur,\n            "domain": r.get("domain", "general"),\n        })\n    return out\n\n\ndef load_audio(path: str) -> np.ndarray:\n    audio, sr = sf.read(path, dtype="float32")\n    if sr != 16000:\n        raise ValueError(f"Expected 16 kHz audio, got {sr} Hz for {path}")\n    return audio if audio.ndim == 1 else audio.mean(axis=1)\n\n\nclass SpeechDataset(Dataset):\n    def __init__(self, rows: list[dict], feature_extractor, tokenizer, with_labels: bool = True) -> None:\n        self.rows = rows\n        self.fe = feature_extractor\n        self.tok = tokenizer\n        self.with_labels = with_labels\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, idx: int) -> dict:\n        row = self.rows[idx]\n        feats = self.fe(load_audio(row["wav"]), sampling_rate=16000).input_features[0]\n        item = {"input_features": feats}\n        if self.with_labels:\n            item["labels"] = self.tok(row["target"]).input_ids\n        return item\n\n\ndef make_collate(processor):\n    def collate(items: list[dict]) -> dict:\n        batch = processor.pad(\n            [{"input_features": x["input_features"]} for x in items],\n            padding=True,\n            return_tensors="pt",\n        )\n        if "labels" in items[0]:\n            labels = processor.pad(\n                labels=[{"input_ids": x["labels"]} for x in items],\n                padding=True,\n                return_tensors="pt",\n            )\n            batch["labels"] = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)\n        return batch\n\n    return collate\n\n\ndef length_batches(durations: list[float], batch_size: int, shuffle: bool) -> list[list[int]]:\n    order = np.argsort(durations)\n    batches = [order[i : i + batch_size].tolist() for i in range(0, len(order), batch_size)]\n    if shuffle:\n        random.shuffle(batches)\n    return batches\n\n\ndef prune_encoder_layers(model: Wav2Vec2BertForCTC, num_layers: int) -> None:\n    """Keep the first `num_layers` encoder layers for frugal Student models."""\n    total = len(model.wav2vec2_bert.encoder.layers)\n    if num_layers < 1 or num_layers > total:\n        raise ValueError(f"--num-layers must be between 1 and {total}, got {num_layers}")\n    if num_layers < total:\n        model.wav2vec2_bert.encoder.layers = model.wav2vec2_bert.encoder.layers[:num_layers]\n        model.config.num_hidden_layers = num_layers\n        print(f"Pruned encoder layers: {total} -> {num_layers}")\n\n\n@torch.no_grad()\ndef evaluate_model(\n    model: Wav2Vec2BertForCTC,\n    processor: Wav2Vec2BertProcessor,\n    val_rows: list[dict],\n    device: str,\n    batch_size: int = 8,\n    use_amp: bool = True,\n) -> dict:\n    model.eval()\n    ds = SpeechDataset(val_rows, processor.feature_extractor, processor.tokenizer, with_labels=False)\n    collate = make_collate(processor)\n    batches = length_batches([r["duration"] for r in val_rows], batch_size, shuffle=False)\n    preds = [""] * len(val_rows)\n\n    for idxs in batches:\n        batch = collate([ds[i] for i in idxs])\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp and device == "cuda"):\n            logits = model(\n                input_features=batch["input_features"].to(device),\n                attention_mask=batch["attention_mask"].to(device),\n            ).logits\n        texts = processor.batch_decode(logits.argmax(-1).cpu(), skip_special_tokens=True)\n        for i, txt in zip(idxs, texts):\n            preds[i] = txt\n\n    gen_pairs = [(r["transcription"], p) for r, p in zip(val_rows, preds) if r["domain"] == "general"]\n    rel_pairs = [(r["transcription"], p) for r, p in zip(val_rows, preds) if r["domain"] == "religious"]\n    res: dict[str, object] = {"predictions": preds}\n\n    if gen_pairs and rel_pairs:\n        mg = measure(gen_pairs)\n        mr = measure(rel_pairs)\n        res["TSER_general"] = mg["TSER"]\n        res["TSER_religious"] = mr["TSER"]\n        res["TER_cond_general"] = mg["TER_conditional"]\n        res["TER_cond_religious"] = mr["TER_conditional"]\n        res["TSER_mirror"] = TEST_WEIGHT_GENERAL * mg["TSER"] + TEST_WEIGHT_RELIGIOUS * mr["TSER"]\n    else:\n        mo = measure([(r["transcription"], p) for r, p in zip(val_rows, preds)])\n        res["TSER_mirror"] = mo["TSER"]\n        res["TER_cond"] = mo["TER_conditional"]\n    return res\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Train CTC Tonal ASR (w2v-bert-2.0 full or layer-pruned).")\n    ap.add_argument("--data-dir", type=Path, required=True, help="Root folder containing extracted audio/")\n    ap.add_argument("--train-csv", type=Path, required=True, help="Primary train CSV (e.g. splits/train_clean.csv)")\n    ap.add_argument("--extra-train-csv", type=Path, help="Optional extra train CSV (e.g. filtered Train_uncertain)")\n    ap.add_argument("--val-csv", type=Path, required=True, help="Validation CSV (e.g. splits/val_mirror.csv)")\n    ap.add_argument("--vocab", type=Path, required=True, help="CTC vocab.json")\n    ap.add_argument("--out-dir", type=Path, required=True, help="Output directory for checkpoints and logs")\n    ap.add_argument("--base-model", default="facebook/w2v-bert-2.0", help="Commercial-licence HF backbone")\n    ap.add_argument("--resume-from", type=Path, help="Resume model weights from a saved checkpoint directory")\n    ap.add_argument("--num-layers", type=int, default=24, help="Number of encoder layers to keep (24, 16, 12, 8)")\n    ap.add_argument("--max-steps", type=int, default=6000)\n    ap.add_argument("--eval-steps", type=int, default=500)\n    ap.add_argument("--eval-limit", type=int, default=600, help="Stratified val cap during training (0 = full val)")\n    ap.add_argument("--batch-size", type=int, default=4)\n    ap.add_argument("--grad-accum", type=int, default=8, help="Effective batch = batch_size * grad_accum")\n    ap.add_argument("--lr", type=float, default=5e-5)\n    ap.add_argument("--weight-decay", type=float, default=0.01)\n    ap.add_argument("--warmup-ratio", type=float, default=0.1)\n    ap.add_argument("--max-seconds", type=float, default=16.0)\n    ap.add_argument("--spec-augment", action="store_true", help="Enable tonal-safe SpecAugment (no pitch shift)")\n    ap.add_argument("--num-workers", type=int, default=2)\n    ap.add_argument("--seed", type=int, default=2026)\n    a = ap.parse_args()\n\n    random.seed(a.seed)\n    np.random.seed(a.seed)\n    torch.manual_seed(a.seed)\n    a.out_dir.mkdir(parents=True, exist_ok=True)\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))\n\n    train_rows = read_rows(a.train_csv, a.data_dir, max_seconds=a.max_seconds, require_target=True)\n    if a.extra_train_csv:\n        extra = read_rows(a.extra_train_csv, a.data_dir, max_seconds=a.max_seconds, require_target=True)\n        print(f"Added {len(extra):,} utterances from {a.extra_train_csv}")\n        train_rows.extend(extra)\n    val_rows = read_rows(a.val_csv, a.data_dir, max_seconds=None, require_target=False)\n\n    if not train_rows:\n        sys.exit(f"No training WAVs found under {a.data_dir}. Extract audio archives first!")\n    if not val_rows:\n        sys.exit(f"No validation WAVs found under {a.data_dir}.")\n\n    if a.eval_limit and len(val_rows) > a.eval_limit:\n        # Stratified sample preserving general/religious domains\n        rng = random.Random(a.seed)\n        gen = [r for r in val_rows if r["domain"] == "general"]\n        rel = [r for r in val_rows if r["domain"] == "religious"]\n        if gen and rel:\n            n_gen = int(round(a.eval_limit * TEST_WEIGHT_GENERAL))\n            n_rel = a.eval_limit - n_gen\n            val_eval = rng.sample(gen, min(n_gen, len(gen))) + rng.sample(rel, min(n_rel, len(rel)))\n        else:\n            val_eval = rng.sample(val_rows, a.eval_limit)\n    else:\n        val_eval = val_rows\n\n    trn_h = sum(r["duration"] for r in train_rows) / 3600.0\n    print(f"Train available: {len(train_rows):,} utts ({trn_h:.2f} h)")\n    print(f"Val available  : {len(val_rows):,} utts (scoring {len(val_eval):,} during training)")\n\n    tokenizer = Wav2Vec2CTCTokenizer(\n        str(a.vocab),\n        unk_token="[UNK]",\n        pad_token="[PAD]",\n        word_delimiter_token="|",\n        bos_token=None,\n        eos_token=None,\n    )\n    fe_src = str(a.resume_from) if a.resume_from else a.base_model\n    feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(fe_src)\n    processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)\n\n    if a.resume_from:\n        print(f"Resuming weights from {a.resume_from}")\n        model = Wav2Vec2BertForCTC.from_pretrained(str(a.resume_from))\n    else:\n        model = Wav2Vec2BertForCTC.from_pretrained(\n            a.base_model,\n            attention_dropout=0.05 if a.spec_augment else 0.0,\n            hidden_dropout=0.05 if a.spec_augment else 0.0,\n            feat_proj_dropout=0.0,\n            mask_time_prob=0.05 if a.spec_augment else 0.0,\n            mask_feature_prob=0.04 if a.spec_augment else 0.0,\n            layerdrop=0.0,\n            ctc_loss_reduction="mean",\n            ctc_zero_infinity=True,\n            add_adapter=True,\n            pad_token_id=tokenizer.pad_token_id,\n            vocab_size=len(tokenizer),\n        )\n        prune_encoder_layers(model, a.num_layers)\n\n    model.gradient_checkpointing_enable()\n    model.to(device)\n    n_params = sum(p.numel() for p in model.parameters()) / 1e6\n    print(f"Model parameters: {n_params:.1f} M ({model.config.num_hidden_layers} encoder layers)")\n\n    (a.out_dir / "training_config.json").write_text(\n        json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in vars(a).items()}, indent=2),\n        encoding="utf-8",\n    )\n\n    train_ds = SpeechDataset(train_rows, feature_extractor, tokenizer, with_labels=True)\n    collate = make_collate(processor)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.weight_decay)\n    warmup_steps = int(a.max_steps * a.warmup_ratio)\n    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, a.max_steps)\n    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")\n\n    step, micro, best_tser = 0, 0, float("inf")\n    running_loss: list[float] = []\n    history: list[dict] = []\n    t0 = time.time()\n\n    model.train()\n    while step < a.max_steps:\n        sampler = length_batches([r["duration"] for r in train_rows], a.batch_size, shuffle=True)\n        loader = DataLoader(train_ds, batch_sampler=sampler, collate_fn=collate, num_workers=a.num_workers)\n        for batch in loader:\n            batch = {k: v.to(device) for k, v in batch.items()}\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):\n                loss = model(**batch).loss / a.grad_accum\n            scaler.scale(loss).backward()\n            running_loss.append(loss.item() * a.grad_accum)\n            micro += 1\n            if micro % a.grad_accum:\n                continue\n\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n            scheduler.step()\n            step += 1\n\n            if step % 100 == 0:\n                avg_l = float(np.mean(running_loss))\n                lr_now = scheduler.get_last_lr()[0]\n                print(\n                    f"step {step:5d}/{a.max_steps} | loss {avg_l:.4f} | lr {lr_now:.2e} | {(time.time() - t0) / 60:.1f} min",\n                    flush=True,\n                )\n                running_loss = []\n\n            if step % a.eval_steps == 0 or step == a.max_steps:\n                metrics = evaluate_model(model, processor, val_eval, device=device)\n                tser = float(metrics["TSER_mirror"])\n                g_str = f" gen={metrics[\'TSER_general\'] * 100:.2f}%" if "TSER_general" in metrics else ""\n                r_str = f" rel={metrics[\'TSER_religious\'] * 100:.2f}%" if "TSER_religious" in metrics else ""\n                print(f"  [EVAL step {step}] TSER_mirror={tser * 100:.2f}%{g_str}{r_str}", flush=True)\n                entry = {"step": step, **{k: v for k, v in metrics.items() if k != "predictions"}}\n                history.append(entry)\n                (a.out_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n\n                # Always save latest for resumability\n                latest_dir = a.out_dir / "latest"\n                model.save_pretrained(latest_dir)\n                processor.save_pretrained(latest_dir)\n\n                if tser < best_tser:\n                    best_tser = tser\n                    best_dir = a.out_dir / "best"\n                    model.save_pretrained(best_dir)\n                    processor.save_pretrained(best_dir)\n                    with open(a.out_dir / "best_val_predictions.csv", "w", encoding="utf-8", newline="") as f:\n                        w = csv.writer(f)\n                        w.writerow(["id", "transcription"])\n                        for r, p in zip(val_eval, metrics["predictions"]):\n                            w.writerow([r["id"], p])\n                    print(f"  >>> New best checkpoint saved to {best_dir} (TSER_mirror={best_tser * 100:.2f}%)", flush=True)\n\n                model.train()\n\n            if step >= a.max_steps:\n                break\n\n    print(f"Done. Best TSER_mirror = {best_tser * 100:.2f}% -> {a.out_dir / \'best\'}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/export_onnx_int8.py': '#!/usr/bin/env python\n"""Export a Wav2Vec2BertForCTC checkpoint to ONNX INT8 for frugal 1-thread CPU inference.\n\nFeatures:\n  - Exports CTC model to ONNX FP32 -> dynamic INT8 quantization (`model_int8.onnx`), then deletes FP32\n  - Copies only minimal offline inference files (`model_int8.onnx`, `vocab.json`) to `--out-dir`\n  - Reports exact size in decimal MB (`bytes / 1e6`) vs the eliminatory 1,000 MB limit\n  - Verifies pure-NumPy feature extractor parity against Hugging Face `SeamlessM4TFeatureExtractor`\n  - Optional `--export-isolated-encoder` to produce `encoder_int8.onnx` for official verification\n  - Optional `--dry-run-pretrained` to benchmark 24L / 16L / 12L / 8L CPU frugality BEFORE training!\n\nUsage (benchmark BEFORE training):\n    python src/export_onnx_int8.py --dry-run-pretrained --num-layers 12 --vocab splits/vocab.json --out-dir weights/bench_12L\n\nUsage (after training):\n    python src/export_onnx_int8.py --model-dir outputs/w2vbert_12L/best --out-dir weights/w2vbert_12L_int8 --export-isolated-encoder\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport shutil\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom onnxruntime.quantization import QuantType, quantize_dynamic\nfrom transformers import SeamlessM4TFeatureExtractor, Wav2Vec2BertForCTC, Wav2Vec2CTCTokenizer\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nfrom src.features_numpy import extract_w2vbert_features  # noqa: E402\n\nMAX_SIZE_BYTES = 1000 * 10**6  # 1,000 MB decimal (measure_frugality.py)\n\n\nclass _CTCWrapper(torch.nn.Module):\n    def __init__(self, model: Wav2Vec2BertForCTC) -> None:\n        super().__init__()\n        self.model = model\n\n    def forward(self, input_features: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        return self.model(input_features=input_features, attention_mask=attention_mask).logits\n\n\nclass _EncoderWrapper(torch.nn.Module):\n    def __init__(self, model: Wav2Vec2BertForCTC) -> None:\n        super().__init__()\n        self.encoder = model.wav2vec2_bert\n\n    def forward(self, input_features: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        return self.encoder(input_features=input_features, attention_mask=attention_mask).last_hidden_state\n\n\ndef check_feature_parity(fe_source: str) -> None:\n    """Verify pure-NumPy feature extractor matches SeamlessM4TFeatureExtractor."""\n    rng = np.random.default_rng(42)\n    wav = (0.1 * rng.standard_normal(16000 * 4)).astype(np.float32)\n    hf_fe = SeamlessM4TFeatureExtractor.from_pretrained(fe_source)\n    ref = hf_fe(wav, sampling_rate=16000).input_features[0]\n    ours = extract_w2vbert_features(wav, sample_rate=16000)\n    min_t = min(ref.shape[0], ours.shape[0])\n    max_diff = float(np.max(np.abs(ref[:min_t] - ours[:min_t])))\n    mean_diff = float(np.mean(np.abs(ref[:min_t] - ours[:min_t])))\n    print(f"Feature parity check: ref={ref.shape}, numpy={ours.shape}, mean_diff={mean_diff:.6f}, max_diff={max_diff:.6f}")\n    if ref.shape != ours.shape or mean_diff > 5e-3:\n        print("WARNING: Feature mismatch detected! Inspect src/features_numpy.py before final inference.", file=sys.stderr)\n\n\ndef export_and_quantize(wrapper: torch.nn.Module, out_onnx: Path, opset: int = 17) -> None:\n    wrapper.eval().cpu()\n    dummy_feats = torch.randn(1, 150, 160, dtype=torch.float32)\n    dummy_mask = torch.ones(1, 150, dtype=torch.long)\n\n    with tempfile.TemporaryDirectory(prefix="onnx_fp32_") as tmp:\n        fp32_path = Path(tmp) / "model_fp32.onnx"\n        export_kwargs = dict(\n            input_names=["input_features", "attention_mask"],\n            output_names=["output"],\n            dynamic_axes={\n                "input_features": {0: "batch", 1: "time"},\n                "attention_mask": {0: "batch", 1: "time"},\n                "output": {0: "batch", 1: "out_time"},\n            },\n            opset_version=opset,\n            do_constant_folding=True,\n        )\n        try:\n            torch.onnx.export(wrapper, (dummy_feats, dummy_mask), str(fp32_path), dynamo=False, **export_kwargs)\n        except TypeError:\n            torch.onnx.export(wrapper, (dummy_feats, dummy_mask), str(fp32_path), **export_kwargs)\n        print(f"Exported FP32 ONNX -> quantizing to INT8...")\n        quantize_dynamic(\n            model_input=str(fp32_path),\n            model_output=str(out_onnx),\n            weight_type=QuantType.QInt8,\n        )\n    print(f"Wrote INT8 ONNX: {out_onnx} ({out_onnx.stat().st_size / 1e6:.1f} MB)")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Export Wav2Vec2BertForCTC to ONNX INT8.")\n    ap.add_argument("--model-dir", type=Path, help="Trained HF checkpoint directory")\n    ap.add_argument("--dry-run-pretrained", action="store_true", help="Load pretrained base model to benchmark frugality")\n    ap.add_argument("--base-model", default="facebook/w2v-bert-2.0")\n    ap.add_argument("--num-layers", type=int, default=24, help="Encoder layers to keep (only for --dry-run-pretrained)")\n    ap.add_argument("--vocab", type=Path, help="vocab.json (required for --dry-run-pretrained)")\n    ap.add_argument("--out-dir", type=Path, required=True, help="Minimal offline weights directory")\n    ap.add_argument("--export-isolated-encoder", action="store_true", help="Also export encoder_int8.onnx outside out-dir")\n    ap.add_argument("--opset", type=int, default=17)\n    a = ap.parse_args()\n\n    a.out_dir.mkdir(parents=True, exist_ok=True)\n\n    if a.dry_run_pretrained:\n        if not a.vocab:\n            sys.exit("--vocab is required with --dry-run-pretrained")\n        tok = Wav2Vec2CTCTokenizer(\n            str(a.vocab),\n            unk_token="[UNK]",\n            pad_token="[PAD]",\n            word_delimiter_token="|",\n            bos_token=None,\n            eos_token=None,\n        )\n        model = Wav2Vec2BertForCTC.from_pretrained(\n            a.base_model,\n            add_adapter=True,\n            pad_token_id=tok.pad_token_id,\n            vocab_size=len(tok),\n        )\n        total = len(model.wav2vec2_bert.encoder.layers)\n        if a.num_layers < total:\n            model.wav2vec2_bert.encoder.layers = model.wav2vec2_bert.encoder.layers[: a.num_layers]\n            model.config.num_hidden_layers = a.num_layers\n        shutil.copy2(a.vocab, a.out_dir / "vocab.json")\n        check_feature_parity(a.base_model)\n    else:\n        if not a.model_dir:\n            sys.exit("Provide --model-dir or --dry-run-pretrained")\n        model = Wav2Vec2BertForCTC.from_pretrained(str(a.model_dir))\n        vocab_src = a.model_dir / "vocab.json"\n        if not vocab_src.exists() and a.vocab:\n            vocab_src = a.vocab\n        if not vocab_src.exists():\n            sys.exit(f"vocab.json not found in {a.model_dir}; pass --vocab")\n        shutil.copy2(vocab_src, a.out_dir / "vocab.json")\n        check_feature_parity(str(a.model_dir))\n\n    model.eval().cpu()\n    export_and_quantize(_CTCWrapper(model), a.out_dir / "model_int8.onnx", opset=a.opset)\n\n    if a.export_isolated_encoder:\n        enc_path = a.out_dir.parent / f"{a.out_dir.name}_isolated_encoder_int8.onnx"\n        export_and_quantize(_EncoderWrapper(model), enc_path, opset=a.opset)\n        print(f"Isolated encoder saved outside --weights dir (so it doesn\'t count twice): {enc_path}")\n\n    total_bytes = sum(f.stat().st_size for f in a.out_dir.rglob("*") if f.is_file())\n    verdict = "COMPLIANT" if total_bytes <= MAX_SIZE_BYTES else "EXCEEDS 1,000 MB LIMIT!"\n    print(f"Total --weights size ({a.out_dir}): {total_bytes / 1e6:.2f} MB / 1000.00 MB -> {verdict}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/infer_onnx.py': '#!/usr/bin/env python\n"""Ultra-frugal, ZERO-PyTorch, 100% offline 1-thread CPU ONNX INT8 inference.\n\nDesigned specifically to pass `measure_frugality.py`:\n  - Does NOT import `torch` or `transformers` (saves ~400-500 MB RAM + fast startup)\n  - Strict single-thread ONNX Runtime (`intra_op=1`, `inter_op=1`, `ORT_SEQUENTIAL`)\n  - Safe long-audio chunking (`--max-chunk-sec`) to cap peak RAM on 50s+ recordings\n  - Supports greedy CTC decoding OR optional KenLM (`--kenlm`) via `pyctcdecode`\n  - Writes plain Fongbe `id,transcription` (compatible with `measure_frugality.py` & `score.py`)\n  - Optional `--syllabified-output` writes ready-to-upload Zindi `ID,Target` via `syllabify()`\n\nUsage with measure_frugality.py (Linux / Kaggle CPU / WSL2):\n    python measure_frugality.py \\\n        --weights weights/w2vbert_12L_int8/ \\\n        --command "python src/infer_onnx.py --weights weights/w2vbert_12L_int8 --csv {csv} --root {root} --output {output}" \\\n        --csv splits/val_mirror.csv --root /tmp/fongbe --limit 300\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport os\nimport sys\nimport wave\nfrom pathlib import Path\n\n# Enforce single-thread BLAS/OMP before importing numpy/onnxruntime\nfor _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "ONNXRUNTIME_NUM_THREADS"):\n    os.environ.setdefault(_var, "1")\n\nimport numpy as np\nimport onnxruntime as ort\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT))\nfrom src.features_numpy import extract_w2vbert_features  # noqa: E402\nfrom syllabify import syllabify  # noqa: E402\n\n\ndef read_wav_16k(path: Path) -> np.ndarray:\n    """Read 16 kHz PCM WAV using stdlib `wave` (fast, zero extra dependency), with soundfile fallback."""\n    try:\n        with wave.open(str(path), "rb") as w:\n            sr = w.getframerate()\n            n_ch = w.getnchannels()\n            sw = w.getsampwidth()\n            raw = w.readframes(w.getnframes())\n        if sr != 16000 or sw != 2:\n            raise ValueError("fallback")\n        audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0\n        if n_ch > 1:\n            audio = audio.reshape(-1, n_ch).mean(axis=1)\n        return audio\n    except Exception:\n        import soundfile as sf\n\n        audio, sr = sf.read(str(path), dtype="float32")\n        if sr != 16000:\n            raise ValueError(f"Expected 16000 Hz audio, got {sr} Hz for {path}")\n        return audio if audio.ndim == 1 else audio.mean(axis=1)\n\n\ndef greedy_ctc_decode(logits: np.ndarray, id2tok: list[str], pad_id: int, unk_id: int) -> str:\n    """Standard CTC collapse (merge consecutive identical IDs, drop blank/pad)."""\n    ids = np.argmax(logits, axis=-1)\n    chars: list[str] = []\n    prev = -1\n    for i in ids:\n        ii = int(i)\n        if ii != prev and ii != pad_id and ii != unk_id and 0 <= ii < len(id2tok):\n            tok = id2tok[ii]\n            chars.append(" " if tok == "|" else tok)\n        prev = ii\n    return "".join(chars).strip()\n\n\ndef build_kenlm_decoder(vocab: dict[str, int], kenlm_path: Path, alpha: float, beta: float):\n    from pyctcdecode import build_ctcdecoder\n\n    # Order labels strictly by token index 0..V-1\n    labels = [""] * len(vocab)\n    for tok, idx in vocab.items():\n        if tok == "[PAD]":\n            labels[idx] = ""\n        elif tok == "|":\n            labels[idx] = " "\n        elif tok == "[UNK]":\n            labels[idx] = ""\n        else:\n            labels[idx] = tok\n    return build_ctcdecoder(labels=labels, kenlm_model_path=str(kenlm_path), alpha=alpha, beta=beta)\n\n\ndef transcribe_waveform(\n    session: ort.InferenceSession,\n    audio: np.ndarray,\n    id2tok: list[str],\n    pad_id: int,\n    unk_id: int,\n    max_chunk_sec: float,\n    kenlm_decoder=None,\n    beam_width: int = 32,\n) -> str:\n    max_samples = int(max_chunk_sec * 16000)\n    if len(audio) <= max_samples:\n        chunks = [audio]\n    else:\n        chunks = [audio[i : i + max_samples] for i in range(0, len(audio), max_samples) if len(audio[i : i + max_samples]) >= 800]\n\n    parts: list[str] = []\n    for chunk in chunks:\n        feats = extract_w2vbert_features(chunk, sample_rate=16000)[None, :, :]\n        mask = np.ones((1, feats.shape[1]), dtype=np.int64)\n        logits = session.run(None, {"input_features": feats, "attention_mask": mask})[0][0]\n        if kenlm_decoder is not None:\n            text = kenlm_decoder.decode(logits, beam_width=beam_width).strip()\n        else:\n            text = greedy_ctc_decode(logits, id2tok, pad_id, unk_id)\n        if text:\n            parts.append(text)\n    return " ".join(parts).strip()\n\n\ndef _infer_worker(args: tuple) -> list[tuple[str, str]]:\n    """Run bit-exact single-thread ONNX INT8 inference on a shard of rows."""\n    (\n        worker_idx,\n        weights_dir,\n        root_dir,\n        rows_shard,\n        id2tok,\n        pad_id,\n        unk_id,\n        max_chunk_sec,\n        kenlm_path,\n        alpha,\n        beta,\n        beam_width,\n        vocab,\n    ) = args\n    import time\n\n    opts = ort.SessionOptions()\n    opts.intra_op_num_threads = 1\n    opts.inter_op_num_threads = 1\n    opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL\n    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n    session = ort.InferenceSession(\n        str(Path(weights_dir) / "model_int8.onnx"),\n        sess_options=opts,\n        providers=["CPUExecutionProvider"],\n    )\n    kenlm_decoder = build_kenlm_decoder(vocab, Path(kenlm_path), alpha, beta) if kenlm_path else None\n    out: list[tuple[str, str]] = []\n    t0 = time.perf_counter()\n    total = len(rows_shard)\n    for idx, r in enumerate(rows_shard, 1):\n        audio = read_wav_16k(Path(root_dir) / r["audio_path"])\n        text = transcribe_waveform(\n            session,\n            audio,\n            id2tok=id2tok,\n            pad_id=pad_id,\n            unk_id=unk_id,\n            max_chunk_sec=max_chunk_sec,\n            kenlm_decoder=kenlm_decoder,\n            beam_width=beam_width,\n        )\n        out.append((r["id"], text))\n        if idx % 200 == 0 or idx == total:\n            dt = time.perf_counter() - t0\n            print(f"[worker {worker_idx}] {idx}/{total} utts ({dt / 60:.1f} min)", flush=True)\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Frugal 1-thread CPU ONNX INT8 inference.")\n    ap.add_argument("--weights", type=Path, required=True, help="Directory containing model_int8.onnx and vocab.json")\n    ap.add_argument("--csv", type=Path, required=True, help="Input CSV with id, audio_path")\n    ap.add_argument("--root", type=Path, required=True, help="Root folder for audio_path")\n    ap.add_argument("--output", type=Path, required=True, help="Output plain Fongbe CSV (id, transcription)")\n    ap.add_argument("--syllabified-output", type=Path, help="Optional Zindi submission CSV (ID, Target)")\n    ap.add_argument("--kenlm", type=Path, help="Optional KenLM .bin/.arpa language model")\n    ap.add_argument("--alpha", type=float, default=0.5, help="KenLM weight")\n    ap.add_argument("--beta", type=float, default=1.0, help="KenLM word insertion bonus")\n    ap.add_argument("--beam-width", type=int, default=32)\n    ap.add_argument("--max-chunk-sec", type=float, default=25.0, help="Chunk audio longer than this (RAM safety)")\n    ap.add_argument(\n        "--workers",\n        type=int,\n        default=1,\n        help="Parallel 1-thread shard workers for fast offline Test.csv generation (ALWAYS keep 1 for measure_frugality.py)",\n    )\n    a = ap.parse_args()\n\n    vocab: dict[str, int] = json.loads((a.weights / "vocab.json").read_text(encoding="utf-8"))\n    id2tok = [""] * len(vocab)\n    for tok, idx in vocab.items():\n        id2tok[idx] = tok\n    pad_id = vocab["[PAD]"]\n    unk_id = vocab["[UNK]"]\n\n    with open(a.csv, encoding="utf-8-sig", newline="") as f:\n        rows = list(csv.DictReader(f))\n\n    if a.workers <= 1:\n        results = _infer_worker((\n            0,\n            str(a.weights),\n            str(a.root),\n            rows,\n            id2tok,\n            pad_id,\n            unk_id,\n            a.max_chunk_sec,\n            str(a.kenlm) if a.kenlm else None,\n            a.alpha,\n            a.beta,\n            a.beam_width,\n            vocab,\n        ))\n    else:\n        import multiprocessing as mp\n        from concurrent.futures import ProcessPoolExecutor\n\n        n_w = min(a.workers, max(1, len(rows)))\n        # Round-robin sharding balances audio durations across workers\n        shards = [rows[i::n_w] for i in range(n_w)]\n        worker_args = [\n            (\n                w_idx,\n                str(a.weights),\n                str(a.root),\n                shard,\n                id2tok,\n                pad_id,\n                unk_id,\n                a.max_chunk_sec,\n                str(a.kenlm) if a.kenlm else None,\n                a.alpha,\n                a.beta,\n                a.beam_width,\n                vocab,\n            )\n            for w_idx, shard in enumerate(shards)\n        ]\n        by_id: dict[str, str] = {}\n        # Use \'spawn\' context on Linux to avoid ONNX Runtime fork() deadlocks!\n        with ProcessPoolExecutor(max_workers=n_w, mp_context=mp.get_context("spawn")) as ex:\n            for shard_res in ex.map(_infer_worker, worker_args):\n                for uid, txt in shard_res:\n                    by_id[uid] = txt\n        results = [(r["id"], by_id[r["id"]]) for r in rows]\n\n    a.output.parent.mkdir(parents=True, exist_ok=True)\n    with open(a.output, "w", encoding="utf-8", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["id", "transcription"])\n        w.writerows(results)\n\n    if a.syllabified_output:\n        a.syllabified_output.parent.mkdir(parents=True, exist_ok=True)\n        with open(a.syllabified_output, "w", encoding="utf-8", newline="") as f:\n            w = csv.writer(f)\n            w.writerow(["ID", "Target"])\n            for uid, txt in results:\n                w.writerow([uid, syllabify(txt)])\n\n\nif __name__ == "__main__":\n    main()\n', 'src/encoder_interface.py': '"""Isolated audio encoder interface for official verification & tone probing.\n\nCompetition verification rule:\n  "top-ranked teams provide their inference code, weights, a Docker container or a\n   reproducible script, the list of external resources with their licences, and\n   their isolated encoder (for the tone analysis published in the summary paper)."\n\nThis module cleanly isolates the acoustic encoder from the CTC classification head\nfor both PyTorch checkpoints and exported ONNX INT8 models.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport numpy as np\n\nfrom src.features_numpy import extract_w2vbert_features\n\n\nclass IsolatedOnnxEncoder:\n    """Zero-PyTorch isolated encoder running on 1 CPU thread via ONNX Runtime."""\n\n    def __init__(self, encoder_onnx_path: str | Path, num_threads: int = 1) -> None:\n        import onnxruntime as ort\n\n        opts = ort.SessionOptions()\n        opts.intra_op_num_threads = num_threads\n        opts.inter_op_num_threads = num_threads\n        opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL\n        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n        self.session = ort.InferenceSession(\n            str(encoder_onnx_path),\n            sess_options=opts,\n            providers=["CPUExecutionProvider"],\n        )\n\n    def encode(self, waveform: np.ndarray, sample_rate: int = 16000) -> np.ndarray:\n        """Return frame-level encoder hidden states of shape (T_enc, D)."""\n        feats = extract_w2vbert_features(waveform, sample_rate=sample_rate)[None, :, :]\n        mask = np.ones((1, feats.shape[1]), dtype=np.int64)\n        hidden = self.session.run(\n            None,\n            {"input_features": feats, "attention_mask": mask},\n        )[0]\n        return hidden[0]\n\n\nclass IsolatedTorchEncoder:\n    """PyTorch isolated encoder exposing final and intermediate layer states."""\n\n    def __init__(self, model_dir: str | Path, device: str = "cpu") -> None:\n        import torch\n        from transformers import Wav2Vec2BertForCTC\n\n        self.device = device\n        full = Wav2Vec2BertForCTC.from_pretrained(str(model_dir), local_files_only=True)\n        # Keep ONLY the acoustic encoder backbone (no lm_head / CTC projection)\n        self.encoder = full.wav2vec2_bert.eval().to(device)\n        self._torch = torch\n\n    def encode(\n        self,\n        waveform: np.ndarray,\n        sample_rate: int = 16000,\n        output_hidden_states: bool = False,\n    ):\n        feats = extract_w2vbert_features(waveform, sample_rate=sample_rate)[None, :, :]\n        x = self._torch.from_numpy(feats).to(self.device)\n        mask = self._torch.ones((1, feats.shape[1]), dtype=self._torch.long, device=self.device)\n        with self._torch.no_grad():\n            out = self.encoder(\n                input_features=x,\n                attention_mask=mask,\n                output_hidden_states=output_hidden_states,\n            )\n        if output_hidden_states:\n            return tuple(h[0].cpu().numpy() for h in out.hidden_states)\n        return out.last_hidden_state[0].cpu().numpy()\n'}

for rel_path, content in EMBEDDED_FILES.items():
    dst = WORK_DIR / rel_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(content, encoding='utf-8')
print(f'{len(EMBEDDED_FILES)} fichiers déployés dans {WORK_DIR}')

In [ ]:
# 3. Récupérer les CSV + télécharger/extraire les archives audio depuis GCS
import shutil, tarfile, urllib.parse, urllib.request, xml.etree.ElementTree as ET
from pathlib import Path

def list_bucket(bucket_url):
    items, marker = [], ""
    while True:
        url = f"{bucket_url}/" + (f"?marker={urllib.parse.quote(marker)}" if marker else "")
        with urllib.request.urlopen(url, timeout=60) as r:
            root = ET.fromstring(r.read())
        trunc, last = False, ""
        for e in root:
            t = e.tag.split("}")[-1]
            if t == "IsTruncated" and (e.text or "").lower() == "true":
                trunc = True
            elif t == "Contents":
                k, s = "", 0
                for c in e:
                    ct = c.tag.split("}")[-1]
                    if ct == "Key": k = c.text or ""
                    elif ct == "Size": s = int(c.text or 0)
                if k:
                    items.append((k, s)); last = k
        if not trunc or not last:
            break
        marker = last
    return items

items = list_bucket(BUCKET_URL)
bucket_keys = {k: s for k, s in items}
print(f"Bucket GCS: {len(items)} fichiers ({sum(s for _, s in items)/1e9:.2f} GB total)")
for k, s in items:
    print(f"  {s/1e6:8.1f} MB  {k}")

# 3a. Localiser les 4 CSV officiels (dans GCS, /kaggle/input, /kaggle/working ou dossier courant)
REQUIRED_CSVS = ["Train_reliable.csv", "Train_religious.csv", "Test.csv", "SampleSubmission.csv"]
search_roots = [p for p in [DATA_DIR, Path("/kaggle/input"), Path("/kaggle/working"), Path("/content"), Path(".")] if p.exists()]

missing_csvs = []
for csv_name in REQUIRED_CSVS:
    dst = DATA_DIR / csv_name
    if dst.exists():
        continue
    # 1) Vérifier si présent directement dans le bucket GCS
    matching_key = next((k for k in bucket_keys if Path(k).name == csv_name), None)
    if matching_key:
        print(f"Téléchargement {csv_name} depuis GCS...")
        urllib.request.urlretrieve(f"{BUCKET_URL}/{urllib.parse.quote(matching_key)}", str(dst))
        continue
    # 2) Sinon chercher dans /kaggle/input ou local
    found = None
    for sr in search_roots:
        hits = list(sr.rglob(csv_name))
        if hits:
            found = hits[0]
            break
    if found:
        shutil.copy2(found, dst)
        print(f"Copié {csv_name} depuis {found}")
    else:
        missing_csvs.append(csv_name)

if missing_csvs:
    raise FileNotFoundError(
        f"CSV manquants: {missing_csvs}. "
        "Sur Kaggle, clique '+ Add Input' -> 'Upload' (ou crée un petit dataset privé de 10 Mo) "
        "avec Train_reliable.csv, Train_religious.csv, Test.csv et SampleSubmission.csv, puis relance !"
    )

# 3b. Télécharger et extraire les archives audio (.tar) dans DATA_DIR (/tmp/fongbe)
rel_tars = sorted([k for k in bucket_keys if "reliable" in k and k.endswith(".tar")])
bib_tars = sorted([k for k in bucket_keys if "religious" in k and k.endswith(".tar")])
tst_tars = sorted([k for k in bucket_keys if "test" in k and k.endswith(".tar")])

if N_RELIABLE_ARCHIVES > 0:
    rel_tars = rel_tars[:N_RELIABLE_ARCHIVES]
if N_RELIGIOUS_ARCHIVES > 0:
    bib_tars = bib_tars[:N_RELIGIOUS_ARCHIVES]
to_get = rel_tars + bib_tars + (tst_tars if DOWNLOAD_TEST else [])

print(f"\nArchives à télécharger ({len(to_get)}): {to_get}")
for key in to_get:
    marker = DATA_DIR / f".done_{Path(key).name}"
    if marker.exists():
        print(f"  Déjà extrait : {key}")
        continue
    tar_path = DATA_DIR / Path(key).name
    print(f"  Téléchargement {key} ({bucket_keys[key]/1e6:.0f} MB)...", flush=True)
    urllib.request.urlretrieve(f"{BUCKET_URL}/{urllib.parse.quote(key)}", str(tar_path))
    print(f"  Extraction {tar_path.name}...", flush=True)
    with tarfile.open(tar_path) as tar:
        tar.extractall(DATA_DIR)
    tar_path.unlink()  # Libère immédiatement l'espace disque !
    marker.touch()

disk = shutil.disk_usage(DATA_DIR)
print(f"\nEspace disque restant dans {DATA_DIR}: {disk.free / 1e9:.1f} GB")

In [ ]:
# 4. Créer les splits miroir sans fuite (train_clean, val_general, val_religious, val_mirror, vocab.json)
import subprocess, sys

SPLITS_DIR = WORK_DIR / "splits"
subprocess.run(
    [sys.executable, str(WORK_DIR / "src/make_splits.py"),
     "--data-root", str(DATA_DIR), "--out-dir", str(SPLITS_DIR), "--val-book", VAL_BOOK],
    check=True,
)

In [ ]:
# 5. Entraînement GPU (w2v-bert-2.0, CTC caractères NFD + tons séparés)
import subprocess, sys

cmd = [
    sys.executable, str(WORK_DIR / "src/train_ctc.py"),
    "--data-dir", str(DATA_DIR),
    "--train-csv", str(SPLITS_DIR / "train_clean.csv"),
    "--val-csv", str(SPLITS_DIR / "val_mirror.csv"),
    "--vocab", str(SPLITS_DIR / "vocab.json"),
    "--out-dir", str(OUT_DIR),
    "--num-layers", str(NUM_LAYERS),
    "--max-steps", str(MAX_STEPS),
    "--eval-steps", str(EVAL_STEPS),
    "--eval-limit", str(EVAL_LIMIT),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--lr", str(LEARNING_RATE),
    "--max-seconds", str(MAX_SECONDS),
]
if SPEC_AUGMENT:
    cmd.append("--spec-augment")
if RESUME_FROM:
    cmd.extend(["--resume-from", str(RESUME_FROM)])

print("Lancement:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# 6. Export ONNX INT8 + Encodeur isolé (vérification officielle)
import subprocess, sys

subprocess.run(
    [
        sys.executable, str(WORK_DIR / "src/export_onnx_int8.py"),
        "--model-dir", str(OUT_DIR / "best"),
        "--vocab", str(SPLITS_DIR / "vocab.json"),
        "--out-dir", str(WEIGHTS_DIR),
        "--export-isolated-encoder",
    ],
    check=True,
)

In [ ]:
# 7. Vérification officielle measure_frugality.py (1 CPU, offline) + Évaluation miroir complète ONNX INT8
import subprocess, sys
import pandas as pd

# Filtrer val_mirror sur les WAV effectivement téléchargés (évite toute erreur si N_RELIABLE_ARCHIVES > 0)
val_df = pd.read_csv(SPLITS_DIR / "val_mirror.csv")
val_avail = val_df[val_df.audio_path.map(lambda p: (DATA_DIR / p).exists())]
val_avail_csv = SPLITS_DIR / "val_mirror_avail.csv"
val_avail.to_csv(val_avail_csv, index=False)
print(f"Validation disponible sur disque : {len(val_avail):,} / {len(val_df):,} phrases")

# 7a. Test officiel de frugalité sur 100 phrases de val_mirror (1 seul cœur CPU, réseau coupé)
frug_cmd = f"{sys.executable} {WORK_DIR / 'src/infer_onnx.py'} --weights {WEIGHTS_DIR} --csv {{csv}} --root {{root}} --output {{output}} --max-chunk-sec 20.0"
subprocess.run(
    [
        sys.executable, str(WORK_DIR / "measure_frugality.py"),
        "--allow-network",
        "--weights", str(WEIGHTS_DIR),
        "--command", frug_cmd,
        "--csv", str(val_avail_csv),
        "--root", str(DATA_DIR),
        "--limit", "100",
        "--report", str(OUT_DIR / "frugality_val100.json"),
    ],
    check=False,
)

# 7b. Évaluation complète des phrases val_mirror en ONNX INT8 (exactement le modèle soumis !)
val_pred_csv = OUT_DIR / "val_mirror_onnx_int8.csv"
subprocess.run(
    [
        sys.executable, str(WORK_DIR / "src/infer_onnx.py"),
        "--weights", str(WEIGHTS_DIR),
        "--csv", str(val_avail_csv),
        "--root", str(DATA_DIR),
        "--output", str(val_pred_csv),
        "--max-chunk-sec", "20.0",
        "--workers", str(INFER_WORKERS),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, str(WORK_DIR / "src/evaluate.py"),
        "--reference", str(val_avail_csv),
        "--predictions", str(val_pred_csv),
        "--output", str(OUT_DIR / "eval_mirror_onnx_int8.json"),
        "--show-errors", "10",
    ],
    check=True,
)

In [ ]:
# 8. Inférence Test.csv (ONNX INT8 bit-exact) + Syllabification + Vérifications Zindi
import re, subprocess, sys
import pandas as pd

if RUN_TEST_INFERENCE:
    test_plain = OUT_DIR / "test_plain_onnx_int8.csv"
    sub_path = OUT_DIR / f"submission_{RUN_NAME}.csv"
    subprocess.run(
        [
            sys.executable, str(WORK_DIR / "src/infer_onnx.py"),
            "--weights", str(WEIGHTS_DIR),
            "--csv", str(DATA_DIR / "Test.csv"),
            "--root", str(DATA_DIR),
            "--output", str(test_plain),
            "--syllabified-output", str(sub_path),
            "--max-chunk-sec", "20.0",
            "--workers", str(INFER_WORKERS),
        ],
        check=True,
    )

    sample = pd.read_csv(DATA_DIR / "SampleSubmission.csv")
    sub = pd.read_csv(sub_path, keep_default_na=False)
    assert list(sub.columns) == ["ID", "Target"], sub.columns
    assert sub.ID.tolist() == sample.ID.tolist(), "IDs ou ordre différents de SampleSubmission.csv !"
    assert sub.ID.is_unique, "IDs dupliqués !"
    assert all(re.fullmatch(r"[a-zɔɖɛ1-5 ]*", t) for t in sub.Target), "Texte non syllabifié détecté !"
    print(f"SOUMISSION VALIDE : {sub_path}")
    print(f"  Lignes : {len(sub):,} | Vides : {(sub.Target == '').sum()}")
    print(sub.head(10))
else:
    print("RUN_TEST_INFERENCE=False : soumission sautée.")